# 06. 3단계 랜덤포레스트 · 3-B 좌우 정보

05 노트북의 0~2단계 함수들은 `src/official.py` 로 옮겼습니다. 내용은 같습니다.

- **3단계**는 공식 데이터만 씁니다 → 보고서 A층
- **3-B 단계**는 외부데이터(`moldset_labeled.csv`)에서 좌우 정보를 가져옵니다 → 보고서 B층, 출처 명기 대상

진행 방식은 05와 같습니다. **질문에 먼저 답을 적고** 실행하세요.

In [ ]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from src.official import *

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

DATA, CEIL = {}, {}
for prod in ["CN7", "RG3"]:
    lab, unl = load(prod)
    for kind in ["A_쌍층화", "B_시간블록"]:
        lab[kind] = make_folds(lab, kind)
    DATA[prod] = (lab, unl)
    CEIL[prod] = f1_ceiling(lab)["F1"]

RESULTS = []
rng = np.random.RandomState(0)
for prod in ["CN7", "RG3"]:
    RESULTS.append(evaluate(DATA[prod][0], rng.rand(len(DATA[prod][0])),
                            "무작위", prod, CEIL[prod]))
print("이론 상한 F1:", {k: round(v, 3) for k, v in CEIL.items()})

---
## 3. 랜덤포레스트 — 공식 데이터, 24개 변수만

### 알아야 할 사실

**랜덤포레스트**는 결정나무 수백 개를 조금씩 다르게 만들어 투표시키는 모델입니다.
각 나무는 "Filling_Time이 0.8보다 크면 왼쪽" 같은 질문을 반복해서 데이터를 나눕니다.
출력은 "불량에 투표한 나무의 비율" — 이게 점수가 됩니다.

**불량이 1.4%뿐이라 그냥 학습하면** 나무들이 "전부 양품"이라고만 해도 거의 다 맞으니 그쪽으로 기웁니다.
그래서 `class_weight="balanced_subsample"` 로 **불량 한 건에 큰 가중치**를 줍니다. 불량 하나를 틀리면 양품 70개를 틀린 것만큼 벌점을 받게 하는 거예요.

**오버샘플링(불량 행 복제) 대신 가중치를 쓰는 이유** — 복제를 train/test 분할 전에 하면 같은 불량의 복제본이 양쪽에 들어가서 모델이 외운 답을 다시 보게 됩니다. 가이드북 F1 1.00의 원인으로 의심되는 바로 그 방식이에요. 가중치는 데이터를 복제하지 않으니 이 위험이 없습니다.

**쌍둥이 쌍은 train에 둘 다 넣습니다.** 같은 입력에 0과 1이 하나씩 — 모델이 그 입력의 불량 확률을 "반반"으로 배우게 되는데, 그게 데이터가 말하는 진실이니까요.

> **질문 3-1.** `class_weight` 없이 학습하면 test 행들의 점수(불량 확률)는 대체로 어떤 값이 나올까요?
>
> 내 답:

> **질문 3-2.** 쌍둥이 쌍(입력 같음, 하나는 불량)이 train에 들어가면, 랜덤포레스트는 그 입력에 대해 대략 몇의 불량 확률을 배울까요?
>
> 내 답:

> **질문 3-3.** B_시간블록에서 CN7은 불량 17건이 전부 fold 0에 있었습니다. fold 0을 test로 쓸 때 train의 불량은 몇 건이고, 그때 모델은 무엇을 할 수 있을까요?
>
> 내 답:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def oof_rf(lab, feats, fold_col, seed=0):
    """fold마다 나머지로 학습해 그 fold를 채점. 모든 행이 한 번씩 test가 된다(out-of-fold)."""
    X, y, f = lab[feats].values, lab["y"].values, lab[fold_col].values
    score = np.full(len(lab), np.nan)
    log = []
    for k in range(K):
        tr, te = f != k, f == k
        npos = int(y[tr].sum())
        if npos == 0:                       # 학습할 불량이 없음 → 모든 행에 같은 점수
            score[te] = 0.0
            log.append({"fold": k, "train 불량": 0, "test 불량": int(y[te].sum()), "비고": "학습 불가"})
            continue
        m = RandomForestClassifier(n_estimators=500, min_samples_leaf=3,
                                   class_weight="balanced_subsample",
                                   n_jobs=-1, random_state=seed)
        m.fit(X[tr], y[tr])
        score[te] = m.predict_proba(X[te])[:, 1]
        log.append({"fold": k, "train 불량": npos, "test 불량": int(y[te].sum()), "비고": ""})
    return score, pd.DataFrame(log)

LOGS = {}
for prod in ["CN7", "RG3"]:
    lab = DATA[prod][0]
    for split in ["A_쌍층화", "B_시간블록"]:
        s, log = oof_rf(lab, VARS, split)
        lab[f"rf_{split}"] = s
        LOGS[(prod, split)] = log
        RESULTS.append(evaluate(lab, s, "랜덤포레스트(24변수)", prod, CEIL[prod], split))

cols = ["모델", "제품", "분할", "AP", "무작위 대비", "F1(최적,참고)", "상한 대비", "상위1%", "상위5%", "상위10%"]
pd.DataFrame(RESULTS)[cols].round(3)

In [ ]:
# fold별로 무슨 일이 있었나
for key, log in LOGS.items():
    print(f"\n── {key[0]} / {key[1]} ──")
    print(log.to_string(index=False))

### 확인
- 질문 3-3의 예상이 맞았는지 — CN7 시간블록에서 "학습 불가" fold가 보이는지
- **A와 B의 차이**: A(같은 시간대가 train/test에 섞임)에서 점수가 높고 B에서 무너진다면,
  모델이 배운 건 "불량의 조건"이 아니라 "불량이 몰렸던 시간대의 특징"일 가능성이 큽니다
- RG3는 분할 방식과 상관없이 무작위 근처일 것 — 쌍둥이 25쌍 때문

> **질문 3-4.** A 분할 점수가 B보다 훨씬 높게 나왔다면, 보고서에는 어느 쪽 숫자를 대표값으로 써야 할까요? 이유는?
>
> 내 답:

---
## 3-B. 좌우 정보 추가 — [외부데이터 사용]

### 알아야 할 사실

공식 파일에는 좌우(LH/RH) 정보가 없습니다. 외부데이터 `moldset_labeled.csv`(KAMP 공개본)에서 가져옵니다.
- 공식 파일의 행 번호(`order`)와 원본의 `Unnamed: 0` 이 **1:1 대응** (라벨 불일치 0 확인)
- 모든 쌍이 LH 하나 + RH 하나로 갈림 (CN7 570/570, RG3 574/574)

좌우를 입력에 넣으면 쌍둥이의 두 행이 **서로 다른 입력**이 됩니다. 논리적으로 구분이 가능해지는 거죠.
그런데 **"구분이 가능하다"와 "실제로 맞힌다"는 다릅니다.** 이 차이를 재는 게 이 단계의 목적입니다.

> **질문 3B-1.** 좌우를 입력에 넣으면 이론 상한 F1은 어떻게 바뀔까요? 1.0까지 갈까요, 아니면 어딘가에서 막힐까요?
>
> 내 답:

> **질문 3B-2.** 모델 없이 "RH면 의심, LH면 안심"이라는 규칙 하나로 점수를 매기면, RG3의 상위 10%·50% 검출률은 얼마일까요? (RH가 전체의 절반, RG3 불량은 전부 RH)
>
> 내 답:

In [ ]:
SIDE, CEIL_SIDE = {}, {}
for prod in ["CN7", "RG3"]:
    lab = attach_side(DATA[prod][0])        # 외부데이터에서 좌우 붙이기 (불일치 시 에러)
    SIDE[prod] = lab
    CEIL_SIDE[prod] = f1_ceiling(lab, VARS + ["is_RH"])["F1"]
    print(f"{prod}: 상한 F1  24변수 {CEIL[prod]:.3f}  →  24변수+좌우 {CEIL_SIDE[prod]:.3f}")
    print(pd.crosstab(lab["side"], lab["y"], margins=True).to_string(), "\n")

In [ ]:
for prod in ["CN7", "RG3"]:
    lab = SIDE[prod]

    # (1) 규칙: RH면 1, LH면 0 — 모델 없음
    RESULTS.append(evaluate(lab, lab["is_RH"].values.astype(float),
                            "규칙: RH만 의심 [외부]", prod, CEIL_SIDE[prod]))

    # (2) 랜덤포레스트 + 좌우
    for split in ["A_쌍층화", "B_시간블록"]:
        s, _ = oof_rf(lab, VARS + ["is_RH"], split)
        lab[f"rfside_{split}"] = s
        RESULTS.append(evaluate(lab, s, "랜덤포레스트+좌우 [외부]", prod, CEIL_SIDE[prod], split))

cols = ["모델", "제품", "분할", "AP", "무작위 대비", "F1(최적,참고)", "상한", "상한 대비",
        "상위1%", "상위5%", "상위10%"]
R = pd.DataFrame(RESULTS)[cols].round(3)
R.sort_values(["제품", "모델", "분할"])

### 확인

같은 제품끼리 세 줄을 비교해보세요.

| 비교 | 무엇을 알려주나 |
|---|---|
| 랜덤포레스트(24변수) → +좌우 | 좌우 정보 한 줄이 실제 성능을 얼마나 올리나 |
| 상한 → 상한(좌우) | 좌우 정보가 **이론적으로** 열어준 여지 |
| 규칙(RH만) vs 랜덤포레스트+좌우 | 공정변수가 좌우 정보 **위에** 더해주는 게 있나 |

> **질문 3B-3.** 상한은 크게 올랐는데 실제 성능은 조금만 올랐다면, 그 차이는 무엇을 뜻할까요? 보고서에 어떻게 쓰면 좋을까요?
>
> 내 답:

> **질문 3B-4.** "RH만 검사" 규칙이 공식 데이터에서 좋아 보여도, 공식 데이터에서 빠진 11/04(RG3, 외부데이터)에는 LH 불량이 5건 있었습니다. 이 사실은 규칙의 신뢰성에 대해 무엇을 말해주나요?
>
> 내 답:

---
## 여기까지 하고 멈춤

질문 답과 마지막 결과표를 공유해주세요. 4단계(직전 N쇼트 이탈도)로 넘어갑니다.